# 02 設定の選択（開発期間）と検証年での成績
3 成分（AoD・even check・外来移動平均）の設定を組として選びます。条件は、開発期間（2023–2024 年）の警報が 0.25 件/週/項目以下で、
1 × TEa の誤差を最も早く検出する組です。比べる構成（2 成分、入院＋外来、全患者の移動平均）も同じ方法で選びます。
最終報告値での選択（感度分析）と、事前の絞り込みなしの総当たり（確認）も実行します。
所要時間の目安：デモ 5 分。実データはデータ量に比例して長くなります（目安 15〜40 分）。

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "pbrtqc").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
pd.set_option("display.max_columns", 30, "display.width", 200)
from pbrtqc import config as C, pipeline
print(C.describe())

> **いまのモード**は上のセルの `profile=` で確認してください（`demo` = 合成データ、`real` = 実データ）。モードを変えたら、VS Code の「Restart」でカーネルを再起動してから上から実行し直します。

In [ ]:
log = pipeline.run([s[0] for s in pipeline.steps() if s[1] == "select"], log_name="run_log_select.json")

## 選ばれた設定（主解析、上限 0.25 件/週）

In [ ]:
p = pd.read_csv(C.TABLES / "select_pairs.csv")
p[(p["cap"].astype(str) == "0.25") & (p["system"] == "aod+even+out")][["item", "setting", "far_dev_wk", "med_hours_dev"]].round(2)

## 検証年に「成立」（1 × TEa の検出時間の中央値が 24 時間以内）した項目の数

In [ ]:
v = pd.read_csv(C.TABLES / "validate.csv")
v1 = v[v["k"] == 1.0]
v1.assign(established=v1["med_hours"] <= 24).pivot_table(index="stream", columns="cap", values="established", aggfunc="sum")

## 事前の絞り込みあり／なしの比較（すべて一致するはず）

In [ ]:
pc = pd.read_csv(C.TABLES / "prune_check.csv")
print(f"{pc['same_setting'].sum()} / {len(pc)} 件で同じ設定")